In [1]:
#!pip install transformers evaluate sentencepiece accelerate pandas==2
#!pip install scikit-learn
#!pip install pyarrow

In [72]:
from datasets import load_dataset
import torch
import datasets
print(datasets.__version__)

2.16.0


In [73]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.__version__)
rand = torch.rand(10, device='cuda')
print(torch.cuda.max_memory_cached(device=None))
print(torch.cuda.memory_allocated(device=None))

True
1
2.4.1
45258637312
2304117760


/local/tmp.3310813/ipykernel_485815/234310295.py:5: FutureWarning: `torch.cuda.max_memory_cached` has been renamed to `torch.cuda.max_memory_reserved`
  print(torch.cuda.max_memory_cached(device=None))


In [74]:
dataset = load_dataset('knowledgator/events_classification_biotech') 
    
classes = [class_ for class_ in dataset['train'].features['label 1'].names if class_]
class2id = {class_:id for id, class_ in enumerate(classes)}
id2class = {id:class_ for class_, id in class2id.items()}

/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/RL/lib/python3.8/site-packages/datasets/load.py:1429: FutureWarning: The repository for knowledgator/events_classification_biotech contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/knowledgator/events_classification_biotech
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [75]:
print('dataset :',dataset)
print('train dataset :', dataset['train'])

print('classes: ',classes)

dataset : DatasetDict({
    train: Dataset({
        features: ['title', 'content', 'target organization', 'all_labels', 'all_labels_concat', 'label 1', 'label 2', 'label 3', 'label 4', 'label 5'],
        num_rows: 2759
    })
    test: Dataset({
        features: ['title', 'content', 'target organization', 'all_labels', 'all_labels_concat', 'label 1', 'label 2', 'label 3', 'label 4', 'label 5'],
        num_rows: 381
    })
})
train dataset : Dataset({
    features: ['title', 'content', 'target organization', 'all_labels', 'all_labels_concat', 'label 1', 'label 2', 'label 3', 'label 4', 'label 5'],
    num_rows: 2759
})
classes:  ['event organization', 'executive statement', 'regulatory approval', 'hiring', 'foundation', 'closing', 'partnerships & alliances', 'expanding industry', 'new initiatives or programs', 'm&a', 'service & product providing', 'new initiatives & programs', 'subsidiary establishment', 'product launching & presentation', 'product updates', 'executive appointment',

In [76]:
from transformers import AutoTokenizer

model_path = 'microsoft/deberta-v3-small'

tokenizer = AutoTokenizer.from_pretrained(model_path)

/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/RL/lib/python3.8/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [77]:
#def split(labels): 
# does not actually properly tokenize the title and content separately it seems! Look at token_type_ids. 
def preprocess_function(example):
    text = f"{example['title']}.\n{example['content']}"
    #print(example['all_labels'])
    all_labels = example['all_labels']
    #print(len(all_labels))
    #if len(all_labels) > 1: 
    #    print(all_labels[0])
    #    all_labels = all_labels.split(',')

    labels = [0. for i in range(len(classes))]
    for label in all_labels:
        label_id = class2id[label]
        labels[label_id] = 1.
  
    example = tokenizer(text, truncation=True)
    example['labels'] = labels
    return example

tokenized_dataset = dataset.map(preprocess_function)

Map:   0%|          | 0/2759 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/381 [00:00<?, ? examples/s]

In [78]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['title', 'content', 'target organization', 'all_labels', 'all_labels_concat', 'label 1', 'label 2', 'label 3', 'label 4', 'label 5', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2759
    })
    test: Dataset({
        features: ['title', 'content', 'target organization', 'all_labels', 'all_labels_concat', 'label 1', 'label 2', 'label 3', 'label 4', 'label 5', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 381
    })
})


In [79]:
import evaluate
import numpy as np

clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def sigmoid(x):
   return 1/(1 + np.exp(-x))

def compute_metrics(eval_pred):
   predictions, labels = eval_pred
   predictions = sigmoid(predictions)
   predictions = (predictions > 0.5).astype(int).reshape(-1)
   return clf_metrics.compute(predictions=predictions, references=labels.astype(int).reshape(-1))

#references=tokenized_dataset['train']['labels'].astype(int).reshape(-1)

In [80]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(

   model_path, num_labels=len(classes),
           id2label=id2class, label2id=class2id,
                       problem_type = "multi_label_classification")


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [81]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [82]:
training_args = TrainingArguments(

   output_dir="my_awesome_model",
   learning_rate=2e-5,
   per_device_train_batch_size=3,
   per_device_eval_batch_size=3,
   num_train_epochs=10,
   weight_decay=0.01,
   evaluation_strategy="epoch",
   save_strategy="epoch",
   load_best_model_at_end=True,
)

trainer = Trainer(

   model=model,
  #  is_model_parallel = True, 
  #  place_model_on_device=True, 
   args=training_args,
   train_dataset=tokenized_dataset["train"],
   eval_dataset=tokenized_dataset["test"],
   tokenizer=tokenizer,	
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

trainer.train()


/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/RL/lib/python3.8/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/local/tmp.3310813/ipykernel_485815/3919293453.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.229600,0.151382,0.946420,0.324201,0.645455,0.216463
2,0.149000,0.127499,0.954928,0.508876,0.720670,0.393293
3,0.120900,0.115006,0.959725,0.605846,0.723044,0.521341
4,0.102200,0.113197,0.959906,0.624258,0.703633,0.560976
5,0.088600,0.107571,0.963617,0.647986,0.761317,0.564024
6,0.076800,0.112501,0.963617,0.657581,0.745174,0.588415
7,0.069400,0.107732,0.966422,0.687447,0.768362,0.621951
8,0.060800,0.109058,0.965879,0.694737,0.740933,0.653963
9,0.053400,0.107076,0.967689,0.707617,0.764602,0.658537
10,0.050400,0.107443,0.967056,0.702128,0.757951,0.653963


TrainOutput(global_step=9200, training_loss=0.09677329187807829, metrics={'train_runtime': 2083.0281, 'train_samples_per_second': 13.245, 'train_steps_per_second': 4.417, 'total_flos': 6744845977136826.0, 'train_loss': 0.09677329187807829, 'epoch': 10.0})

In [13]:
trainer.place_model_on_device
#trainer.is_model_parallel

True

#### Tokenizer output 
The tokenizer outputs the following: (https://huggingface.co/docs/transformers/v4.14.1/en/main_classes/tokenizer)
The objective of the tokenizer is to translate the input into numbers (input IDs). 
Raw text, e.g. "this course is amazing", could be mapped to input IDs [101, 2023, 2607, ...]. The tokenizer's objective is to find the most meaningful representation. There are different ways: word-based, character-based and subword-based. It seems like this tokenizer is word-based. The input_ids are the most important output. 

* token_type_ids: A list of token type ids to be fed to a model.
* attention_mask: List of indices specifying which tokens should be attended to by the model.
* input ids: A list of token ids to be fed to a model.
* labels: This is just the collation of all labels (multilabel classification). We added that in the tokenizer function.

Then, the attention mask determines what tokens should be attended to, and which not. In our case, these just all seem to be ones, so each word is equally important. 

Some models need to do classification on pairs of sentences/questions and answers. Another input that is relevant could be if two questions are duplicate or not (important!). This is also true in the case of this dataset, where there is a title and there is a longer description called content. These are separated as [CLS] Title [SEP] Content [SEP]. 
However, when looking at the training, it looks like it's just the content that is passed on to the model. 

In [44]:
print(tokenized_dataset["test"])
print(dataset["test"])
# take a random idx 
print(len(tokenized_dataset["test"]))
#print(tokenized_dataset["test"][0])
first_sample = tokenized_dataset["test"][0]
# has three fields: title, content and target organization
print(tokenized_dataset["test"][0]['title'])
#print(tokenized_dataset["test"][0]['content'])
print(tokenized_dataset["test"][0]['labels'])

#print(dataset["test"][0]['content'])
print(tokenized_dataset["test"][0]['labels'])
#print(tokenized_dataset["test"][0]['attention_mask'])
print(tokenized_dataset["test"][0]['token_type_ids'])
print(len(tokenized_dataset["test"][0]['input_ids'])) 
#print(len(tokenized_dataset["test"][0]['attention_mask']))
#decoded = tokenizer.decode(tokenized_dataset["test"][0]['input_ids'])
#print(decoded)


Dataset({
    features: ['title', 'content', 'target organization', 'all_labels', 'all_labels_concat', 'label 1', 'label 2', 'label 3', 'label 4', 'label 5', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 381
})
Dataset({
    features: ['title', 'content', 'target organization', 'all_labels', 'all_labels_concat', 'label 1', 'label 2', 'label 3', 'label 4', 'label 5'],
    num_rows: 381
})
381
Renton investor purchases Vivacity Care building
[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [18]:
# visualize the result somehow 
print(model)

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): StableDropout()
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): StableDropout()
              (dropout): StableDropout()
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=T

In [48]:
#print(tokenized_dataset["test"][0])
output = trainer.predict(tokenized_dataset["test"])
print(output.shape)

AttributeError: 'PredictionOutput' object has no attribute 'shape'

In [49]:
print(output.predictions.shape)

(381, 29)


In [64]:
print(output.predictions[0,:])
predictions_0 = torch.topk(torch.tensor(output.predictions[0,:]),3)
print(torch.topk(torch.tensor(output.predictions[0,:]),3))
print(tokenized_dataset["test"][0]['labels'])
idx_0 = torch.topk(torch.tensor(output.predictions[0,:]),3).indices.numpy()
label_0 = tokenized_dataset["test"][0]['labels']
print(label_0[idx_0[0]])
print(label_0[idx_0[1]])
print(label_0[idx_0[2]])
# third one is incorrect. 

[-3.6566336   1.8611205  -3.4842837  -4.1991553  -4.9339285  -5.102802
 -5.491837   -3.9396183  -1.897195   -1.4819976  -2.914874   -3.9164767
 -4.3648157  -2.6287029  -3.1058893  -1.8044547  -0.9216165  -4.637517
 -4.596572   -4.162175   -0.01273195 -3.4687116  -3.8841383  -1.4509205
 -3.640772   -3.4009902  -4.507847   -2.1091015  -5.1784806 ]
torch.return_types.topk(
values=tensor([ 1.8611, -0.0127, -0.9216]),
indices=tensor([ 1, 20, 16]))
[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
1.0
1.0
0.0


In [70]:
def compute_score(pred, label):
   predictions = sigmoid(pred)
   predictions = (predictions > 0.5).astype(int).reshape(-1)
   print(predictions)
   print(len(predictions))
   return clf_metrics.compute(predictions=predictions, references=[int(x) for x in label])
# basically, all classes that are predicted have a value above 0.5. 
compute_score(output.predictions[0,:], label_0)
# accuracy: 27/29 correct = 93%. 

[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
29


{'accuracy': 0.9310344827586207,
 'f1': 0.5,
 'precision': 1.0,
 'recall': 0.3333333333333333}